# Pneumonia Detection Project Guide

Follow these steps to set up and run your Pneumonia Detection Project with a Gradio UI using real clinical data from Kaggle.

### Step-by-Step Execution Plan:
1. **Get Kaggle API Token**: Go to your Kaggle Account page (`https://www.kaggle.com/settings`), scroll down to the **API** section, and click **Create New Token**. This downloads a file named `kaggle.json`.
2. **Upload Kaggle Token to Colab**: Run **Step 0** below to upload your `kaggle.json` file to this notebook environment.
3. **Install Dependencies**: Run **Step 1** to install the required libraries.
4. **Download Dataset**: Run the Kaggle download cells to pull the Chest X-Ray dataset.
5. **Model & Pipeline**: Run **Step 3** to configure the deep learning classification model.
6. **Launch UI**: Run **Step 4** to launch the Gradio user interface and test it.

### Step 0: Upload Kaggle API Key and Download Dataset
Run the cell below to upload your `kaggle.json` file.

In [ ]:
from google.colab import files
import os

# Prompt to upload kaggle.json
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your 'kaggle.json' file:")
    uploaded = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    for fn in uploaded.keys():
        os.rename(fn, '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 600)
    print("Kaggle API key uploaded successfully!")
else:
    print("Kaggle API key is already present.")

Kaggle API key is already present.


Now, we will download the official chest X-ray dataset from Kaggle using the API.

In [ ]:
# Download the dataset zip from Kaggle
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

# Create directory and extract
import zipfile
os.makedirs('chest_xray_data', exist_ok=True)
with zipfile.ZipFile('chest-xray-pneumonia.zip', 'r') as zip_ref:
    zip_ref.extractall('chest_xray_data')
print("Dataset downloaded and extracted!")

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
chest-xray-pneumonia.zip: Skipping, found more recently modified local copy (use --force to force download)
Dataset downloaded and extracted!


### Step 1: Install Required Libraries
We will use `gradio` for building the web interface, and `timm`, `torchvision`, and `torch` to build and run our deep learning model.

In [ ]:
!pip install -q gradio timm

### Step 2: Download or Simulate a Chest X-ray Dataset
For demonstration, we will download a small subset of chest X-ray images or create a mock dataset script to ensure you can run the entire pipeline instantly. Let's write a python script to download sample images for testing.

In [ ]:
import os
import numpy as np
from PIL import Image, ImageDraw

# Create a directory for testing images
os.makedirs('sample_images', exist_ok=True)

def generate_synthetic_xray(filename, label):
    # Generate a gray canvas mimicking a chest X-ray
    img = Image.new('RGB', (224, 224), color='#1a1a1a')
    draw = ImageDraw.Draw(img)

    # Draw simulated lung cavities
    draw.ellipse([30, 40, 95, 200], fill='#333333', outline='#555555', width=2) # Left lung
    draw.ellipse([125, 40, 190, 200], fill='#333333', outline='#555555', width=2) # Right lung

    # Draw spine/mediastinum
    draw.rectangle([98, 30, 122, 210], fill='#888888')

    # Simulate ribs
    for y in range(60, 190, 25):
        draw.line([35, y, 95, y+5], fill='#999999', width=2)
        draw.line([125, y+5, 185, y], fill='#999999', width=2)

    if label == "pneumonia":
        # Simulate hazy lung infiltrates/opacity characteristic of pneumonia
        draw.ellipse([45, 110, 85, 170], fill='#777777') # Hazy consolidation
        draw.ellipse([135, 120, 175, 160], fill='#666666')

    filepath = os.path.join('sample_images', filename)
    img.save(filepath)
    print(f"Generated synthetic sample: {filename} ({label.upper()})")

try:
    generate_synthetic_xray('normal.jpeg', 'normal')
    generate_synthetic_xray('pneumonia.jpeg', 'pneumonia')
except Exception as e:
    print(f"Could not generate sample images: {e}")

Generated synthetic sample: normal.jpeg (NORMAL)
Generated synthetic sample: pneumonia.jpeg (PNEUMONIA)


### Step 3: Define the Model and Prediction Pipeline
Here, we initialize a pre-trained ResNet-18 model using `timm` and set up the image pre-processing pipeline to classify input images into 'Normal' or 'Pneumonia'.

In [ ]:
import torch
import timm
from PIL import Image
import torchvision.transforms as transforms

# Load a pre-trained ResNet18 model and adapt it for 2 classes (Normal vs Pneumonia)
model = timm.create_model('resnet18', pretrained=True, num_classes=2)
model.eval()  # Put model in evaluation mode

# Define image transformation pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def predict_pneumonia(image):
    if image is None:
        return "No image uploaded"

    # Convert Gradio input (numpy array/PIL image) and apply transforms
    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert('RGB')

    input_tensor = transform(image).unsqueeze(0) # Add batch dimension

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Map output probabilities
    classes = ['Normal', 'Pneumonia']
    return {classes[i]: float(probabilities[i]) for i in range(2)}

### Step 3b: Fine-tune the ResNet-18 Model on the Chest X-Ray Dataset
Since our pre-trained model has never seen chest X-rays, we will quickly train (fine-tune) its classification head using the Kaggle dataset we downloaded. This will dramatically improve its prediction accuracy.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. FREEZE BACKBONE FOR INSTANT TRANSFER LEARNING ---
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the classification head
for param in model.get_classifier().parameters():
    param.requires_grad = True

model = model.to(device)

# --- 2. DATASETS & FAST DATA REDUCTION (SUBSAMPLING) ---
train_dir = 'chest_xray_data/chest_xray/train'
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)

# Pick exactly 100 random normal and 100 random pneumonia images to train instantly
targets = np.array(full_train_dataset.targets)
normal_idx = np.where(targets == 0)[0]
pneumonia_idx = np.where(targets == 1)[0]

# Set seed for reproducible quick subsampling
np.random.seed(42)
sub_normal_idx = np.random.choice(normal_idx, 100, replace=False)
sub_pneumonia_idx = np.random.choice(pneumonia_idx, 100, replace=False)

sub_indices = np.concatenate([sub_normal_idx, sub_pneumonia_idx])
subset_train_dataset = Subset(full_train_dataset, sub_indices)

train_loader = DataLoader(subset_train_dataset, batch_size=16, shuffle=True)

# --- 3. STANDARD LOSS & OPTIMIZER ---
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.get_classifier().parameters(), lr=0.001)

# --- 4. FAST 2-EPOCH TRAINING ---
print(f"Training model on a quick, balanced subset of {len(subset_train_dataset)} images...")
model.train()
for epoch in range(2):
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(subset_train_dataset)
    epoch_acc = 100. * correct / total
    print(f"Epoch {epoch+1}/2 - Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

model.eval()
print("Fine-tuning complete! Re-test your Gradio app now.")

Training model on a quick, balanced subset of 200 images...
Epoch 1/2 - Loss: 0.6779 | Accuracy: 60.00%
Epoch 2/2 - Loss: 0.6066 | Accuracy: 85.50%
Fine-tuning complete! Re-test your Gradio app now.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import numpy as np

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. RE-INITIALIZE AND UNFREEZE LATER LAYERS ---
# Load fresh pre-trained resnet18
model = timm.create_model('resnet18', pretrained=True, num_classes=2)

# Freeze all early layers
for param in model.parameters():
    param.requires_grad = False

# Unfreeze layer4 (the last convolutional block) and the classifier
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.get_classifier().parameters():
    param.requires_grad = True

model = model.to(device)

# --- 2. PREPARE BALANCED TRAINING DATA ---
train_dir = 'chest_xray_data/chest_xray/train'
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)

# Pick 150 normal and 150 pneumonia samples for a robust quick dataset
targets = np.array(full_train_dataset.targets)
normal_idx = np.where(targets == 0)[0]
pneumonia_idx = np.where(targets == 1)[0]

np.random.seed(42)
sub_normal_idx = np.random.choice(normal_idx, 150, replace=False)
sub_pneumonia_idx = np.random.choice(pneumonia_idx, 150, replace=False)

sub_indices = np.concatenate([sub_normal_idx, sub_pneumonia_idx])
subset_train_dataset = Subset(full_train_dataset, sub_indices)

train_loader = DataLoader(subset_train_dataset, batch_size=16, shuffle=True)

# --- 3. LOSS & OPTIMIZER WITH SMALLER LEARNING RATE ---
criterion = nn.CrossEntropyLoss()
# Use Adam with a lower learning rate since we are training deeper layers
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4},
    {'params': model.get_classifier().parameters(), 'lr': 1e-3}
])

# --- 4. TRAIN FOR 5 EPOCHS ---
print(f"Training model with fine-tuned backend on {len(subset_train_dataset)} images...")
for epoch in range(5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(subset_train_dataset)
    epoch_acc = 100. * correct / total
    print(f"Epoch {epoch+1}/5 - Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

model.eval()
print("Optimized fine-tuning complete! Now, re-run the Gradio interface cell below to test prediction output.")

Training model with fine-tuned backend on 300 images...
Epoch 1/5 - Loss: 0.6673 | Accuracy: 58.00%
Epoch 2/5 - Loss: 0.5487 | Accuracy: 87.33%
Epoch 3/5 - Loss: 0.4591 | Accuracy: 89.33%
Epoch 4/5 - Loss: 0.3681 | Accuracy: 92.00%
Epoch 5/5 - Loss: 0.3251 | Accuracy: 89.67%
Optimized fine-tuning complete! Now, re-run the Gradio interface cell below to test prediction output.


In [ ]:
# Update our prediction function to use GPU if available
def predict_pneumonia(image):
    if image is None:
        return "No image uploaded"

    if not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert('RGB')

    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    classes = ['Normal', 'Pneumonia']
    return {classes[i]: float(probabilities[i]) for i in range(2)}

### Step 4: Create and Launch the Gradio User Interface
Now we create a web-based dashboard directly within your Colab notebook allowing users to upload an image and view real-time predictions.

In [ ]:
import os
import gradio as gr
import random

# Find real sample images from the downloaded Kaggle dataset to display in the UI
example_list = []

normal_dir = 'chest_xray_data/chest_xray/test/NORMAL'
pneumonia_dir = 'chest_xray_data/chest_xray/test/PNEUMONIA'

if os.path.exists(normal_dir) and os.path.exists(pneumonia_dir):
    # Pick a random normal and a random pneumonia image from the dataset test folder
    normal_samples = [os.path.join(normal_dir, f) for f in os.listdir(normal_dir) if f.endswith(('.jpeg', '.jpg', '.png'))]
    pneumonia_samples = [os.path.join(pneumonia_dir, f) for f in os.listdir(pneumonia_dir) if f.endswith(('.jpeg', '.jpg', '.png'))]

    if normal_samples:
        example_list.append([random.choice(normal_samples)])
    if pneumonia_samples:
        example_list.append([random.choice(pneumonia_samples)])
else:
    # Fallback to synthetic samples if path doesn't exist
    if os.path.exists('sample_images/normal.jpeg'):
        example_list.append(['sample_images/normal.jpeg'])
    if os.path.exists('sample_images/pneumonia.jpeg'):
        example_list.append(['sample_images/pneumonia.jpeg'])

# Create Gradio Interface
interface = gr.Interface(
    fn=predict_pneumonia,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=gr.Label(num_top_classes=2, label="Model Prediction"),
    title="Pneumonia Detection AI Assistant",
    description="Upload a chest X-ray image (JPEG/PNG) or select one of the real test clinical examples below to test the AI model.",
    examples=example_list if example_list else None
)

# Launch UI with queue enabled and inline representation explicitly forced
interface.queue().launch(
    inline=True,
    share=True,
    debug=False,
    show_error=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4a904a58f960b2d49c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
